# ACL Tear Detection — V7 MRI-Only Training

**Problem in V6.x:** Source-label confounding — Priyank ≈ 95% Tear, Kaggle MRI ≈ 80% Normal.
The model learned "which dataset is this?" instead of "is there an ACL tear?"

**V7 fix:** Train exclusively on the MRI/Kaggle dataset (which has both classes in reasonable proportions).
Use the Priyank dataset purely as **external validation** to demonstrate generalization.

**Architecture:**
- **Part A:** Single-split training on MRI data → test on MRI + external eval on Priyank
- **Part B:** 5-Fold CV on MRI data → each fold externally validated on Priyank

| Setting | V6.2 | V7 | Why |
|---------|------|----|-----|
| Training data | MRI + Priyank | **MRI only** | Eliminates source-label confound |
| External validation | ❌ | **Priyank** | Shows generalization to unseen source |
| Sampler | source+class balanced | **class balanced** | Single source, no source balancing needed |
| Backbone frozen | 50% | **70%** | Less data → need more regularization |
| Intensity augmentation | ✅ | **✅** | Still useful for robustness |
| All other settings | — | Same | Keep what works |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
DATA_DIR = '/content/drive/MyDrive/dataset/combined'

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 2  # 0: Normal, 1: Tear (Partial + Complete)
RANDOM_SEED = 42

# Source-aware slice ranges (needed for external Priyank eval)
SLICE_RANGE_PRIYANK = (33, 43)   # Slices 33–42 (inclusive) for Priyank Saxena
SLICE_RANGE_MRI     = (12, 22)   # Slices 12–21 (inclusive) for Kaggle MRI

# CLAHE settings
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID  = (8, 8)

# Percentile normalization
PERCENTILE_LOW  = 2
PERCENTILE_HIGH = 98

# ---- V7 Settings ----
FREEZE_FRACTION = 0.70       # 70% frozen — less MRI-only data needs more regularization
CLASSIFIER_HIDDEN = 192      # Same as V6.x
DROPOUT_1 = 0.45             # Same as V6.x
DROPOUT_2 = 0.35             # Same as V6.x
LABEL_SMOOTHING = 0.1        # Same as V6.x
WEIGHT_DECAY = 0.02          # Same as V6.x
PATIENCE = 10                # Same as V6.2
N_FOLDS = 5                  # K-Fold CV folds

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import cv2
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load Metadata & Split by Source
# ============================================================
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

metadata['label_binary'] = (metadata['label'] > 0).astype(int)
metadata['label_name_binary'] = metadata['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"Total patients: {len(metadata)}")
print(f"\nSource × Label crosstab:")
print(pd.crosstab(metadata['source'], metadata['label_name_binary'], margins=True))

# ---- V7: Split by source ----
mri_df = metadata[metadata['source'] == 'MRI'].reset_index(drop=True)
priyank_df = metadata[metadata['source'] == 'Priyank_Saxena'].reset_index(drop=True)

print(f"\n{'='*50}")
print(f"MRI (Training source): {len(mri_df)} patients")
print(mri_df['label_name_binary'].value_counts())
imbalance_mri = mri_df['label_binary'].value_counts().max() / mri_df['label_binary'].value_counts().min()
print(f"Class imbalance ratio: {imbalance_mri:.1f}x")

print(f"\nPriyank Saxena (External validation): {len(priyank_df)} patients")
print(priyank_df['label_name_binary'].value_counts())
print(f"⚠️  Priyank is heavily skewed toward Tear — external validation results")
print(f"    will primarily test Tear detection on a different MRI source.")

In [ ]:
# ============================================================
# Preprocessing Functions
# ============================================================

def apply_clahe(img, clip_limit=CLAHE_CLIP_LIMIT, tile_grid=CLAHE_TILE_GRID):
    img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(img_uint8).astype(np.float32) / 255.0

def percentile_normalize(img, low=PERCENTILE_LOW, high=PERCENTILE_HIGH):
    p_low, p_high = np.percentile(img, [low, high])
    img = np.clip(img, p_low, p_high)
    denom = p_high - p_low
    if denom > 0:
        img = (img - p_low) / denom
    return img

def random_intensity_augment(img):
    """Random intensity perturbation for robustness.
    Helps the model generalize to unseen MRI sources (like Priyank).
    """
    # Random gamma correction
    gamma = np.random.uniform(0.65, 1.5)
    img = np.power(np.clip(img, 0, 1), gamma)
    
    # Random brightness shift
    brightness = np.random.uniform(-0.12, 0.12)
    img = img + brightness
    
    # Random contrast scaling
    contrast = np.random.uniform(0.75, 1.25)
    mean = img.mean()
    img = (img - mean) * contrast + mean
    
    # Random Gaussian noise
    if np.random.random() < 0.5:
        noise_std = np.random.uniform(0.01, 0.04)
        noise = np.random.normal(0, noise_std, img.shape).astype(np.float32)
        img = img + noise
    
    return np.clip(img, 0, 1).astype(np.float32)

In [ ]:
# ============================================================
# Dataset Class (supports source-aware slice selection)
# ============================================================

class ACLSliceDataset(Dataset):
    def __init__(self, df, data_dir, transform=None,
                 slice_range_priyank=SLICE_RANGE_PRIYANK,
                 slice_range_mri=SLICE_RANGE_MRI,
                 use_clahe=True,
                 augment_intensity=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.slice_range_priyank = slice_range_priyank
        self.slice_range_mri = slice_range_mri
        self.use_clahe = use_clahe
        self.augment_intensity = augment_intensity

        self.samples = []
        skipped = 0
        for idx, row in self.df.iterrows():
            num_slices = int(row['num_slices'])
            source = row.get('source', 'unknown')

            if source == 'Priyank_Saxena':
                start, end = self.slice_range_priyank
            else:
                start, end = self.slice_range_mri

            start = min(start, num_slices)
            end = min(end, num_slices)

            if start >= end:
                skipped += 1
                continue

            for slice_idx in range(start, end):
                self.samples.append((idx, slice_idx, int(row['label_binary']), source))

        if skipped > 0:
            print(f"  ⚠️ Skipped {skipped} patients (slice range out of bounds)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        patient_idx, slice_idx, label, source = self.samples[idx]
        row = self.df.iloc[patient_idx]

        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']
        slice_idx = min(slice_idx, volume.shape[0] - 1)

        img = volume[slice_idx].astype(np.float32) / 255.0
        img = percentile_normalize(img)

        if self.use_clahe:
            img = apply_clahe(img)

        if self.augment_intensity:
            img = random_intensity_augment(img)

        img_mean = img.mean()
        img_std = img.std() + 1e-8
        img = (img - img_mean) / img_std

        img = np.stack([img, img, img], axis=0)
        img = torch.from_numpy(img)

        if self.transform:
            img = self.transform(img)

        return img, label, patient_idx

    def get_labels(self):
        return [s[2] for s in self.samples]

    def get_sources(self):
        return [s[3] for s in self.samples]

In [ ]:
# ============================================================
# Class-Balanced Sampler (V7 — single source, no source balancing)
# ============================================================

def make_class_balanced_sampler(dataset):
    """Class-balanced sampler for single-source training.
    Unlike V6's source+class sampler, this only balances Normal vs Tear.
    """
    labels = dataset.get_labels()
    class_counts = Counter(labels)

    print("  Class distribution:")
    for cls, count in sorted(class_counts.items()):
        print(f"    {'Normal' if cls==0 else 'Tear':6s}: {count}")

    sample_weights = [1.0 / class_counts[label] for label in labels]

    sampler = WeightedRandomSampler(
        sample_weights, len(sample_weights), replacement=True
    )
    return sampler

In [ ]:
# ============================================================
# Data Augmentation
# ============================================================

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = None

In [ ]:
# ============================================================
# Model Factory
# ============================================================

def create_model(freeze_fraction=FREEZE_FRACTION,
                 hidden_size=CLASSIFIER_HIDDEN,
                 dropout1=DROPOUT_1,
                 dropout2=DROPOUT_2):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')

    all_params = list(model.features.parameters())
    freeze_until = int(len(all_params) * freeze_fraction)
    for i, param in enumerate(all_params):
        param.requires_grad = (i >= freeze_until)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout1),
        nn.Linear(in_features, hidden_size),
        nn.ReLU(),
        nn.Dropout(p=dropout2),
        nn.Linear(hidden_size, NUM_CLASSES)
    )

    model = model.to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Backbone: {freeze_fraction*100:.0f}% frozen")
    print(f"  Classifier: {hidden_size} hidden, dropout={dropout1}/{dropout2}")
    print(f"  Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")
    return model

In [ ]:
# ============================================================
# Training & Validation Functions
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels, _ in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_patient_indices = []

    with torch.no_grad():
        for images, labels, patient_idx in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_patient_indices.extend(patient_idx.numpy())

    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels, all_patient_indices

In [ ]:
# ============================================================
# Patient-Level Majority Voting Helper
# ============================================================

def patient_level_metrics(preds, labels, patient_indices, dataset_df=None,
                          label_names=['Normal', 'Tear'], verbose=True):
    patient_preds_dict = {}
    patient_labels_dict = {}

    for pred, label, pidx in zip(preds, labels, patient_indices):
        pidx = int(pidx)
        if pidx not in patient_preds_dict:
            patient_preds_dict[pidx] = []
            patient_labels_dict[pidx] = label
        patient_preds_dict[pidx].append(pred)

    patient_final_preds = []
    patient_final_labels = []

    for pidx in sorted(patient_preds_dict.keys()):
        final_pred = Counter(patient_preds_dict[pidx]).most_common(1)[0][0]
        patient_final_preds.append(final_pred)
        patient_final_labels.append(patient_labels_dict[pidx])

    patient_acc = np.mean(
        np.array(patient_final_preds) == np.array(patient_final_labels)
    ) * 100

    if verbose:
        print(f"\nPatient-level Accuracy: {patient_acc:.1f}% ({len(patient_final_preds)} patients)")
        print(classification_report(
            patient_final_labels, patient_final_preds,
            target_names=label_names, digits=3, zero_division=0
        ))

    return patient_acc, patient_final_preds, patient_final_labels

---
## Part A: Single-Split Training (MRI Only → External Priyank Validation)

In [ ]:
# ============================================================
# Train/Val/Test Split — MRI ONLY
# ============================================================
train_df, temp_df = train_test_split(
    mri_df, test_size=0.3, stratify=mri_df['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print("MRI Dataset Split:")
print(f"  Train: {len(train_df)} patients")
print(f"  Val:   {len(val_df)} patients")
print(f"  Test:  {len(test_df)} patients")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n  {name} class distribution:")
    print(f"    {df['label_name_binary'].value_counts().to_dict()}")

print(f"\nExternal Validation (Priyank Saxena): {len(priyank_df)} patients")
print(f"  {priyank_df['label_name_binary'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# Create Datasets
# ============================================================
print("Creating datasets...")
print(f"  MRI slice range: {SLICE_RANGE_MRI[0]}–{SLICE_RANGE_MRI[1]-1}")
print(f"  Priyank slice range: {SLICE_RANGE_PRIYANK[0]}–{SLICE_RANGE_PRIYANK[1]-1}")
print()

print("Train (MRI, with intensity augmentation):")
train_dataset = ACLSliceDataset(train_df, DATA_DIR, train_transform, augment_intensity=True)
print("Val (MRI):")
val_dataset = ACLSliceDataset(val_df, DATA_DIR, val_transform)
print("Test (MRI):")
test_dataset = ACLSliceDataset(test_df, DATA_DIR, val_transform)
print("External Validation (Priyank):")
priyank_dataset = ACLSliceDataset(priyank_df, DATA_DIR, val_transform)

print(f"\nTrain samples (slices): {len(train_dataset)}")
print(f"Val samples (slices):   {len(val_dataset)}")
print(f"Test samples (slices):  {len(test_dataset)}")
print(f"Priyank samples (slices): {len(priyank_dataset)}")

In [ ]:
# ============================================================
# Class-Balanced Sampler & DataLoaders
# ============================================================
print("\nCreating class-balanced sampler (MRI only)...")
train_sampler = make_class_balanced_sampler(train_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
priyank_loader = DataLoader(priyank_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ============================================================
# Create Model, Loss, Optimizer
# ============================================================
print("\nV7 Model:")
model = create_model()

train_labels = train_dataset.get_labels()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=LABEL_SMOOTHING
)

optimizer = optim.AdamW(
    model.parameters(), lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

print(f"\nV7 settings:")
print(f"  Training on:     MRI/Kaggle ONLY")
print(f"  External val:    Priyank Saxena")
print(f"  Freeze:          {FREEZE_FRACTION*100:.0f}%")
print(f"  Classifier:      {CLASSIFIER_HIDDEN} hidden")
print(f"  Dropout:         {DROPOUT_1}/{DROPOUT_2}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Weight decay:    {WEIGHT_DECAY}")
print(f"  Patience:        {PATIENCE}")
print(f"  Intensity aug:   ✅ ON")

In [ ]:
# ============================================================
# Training Loop
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v7.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels, val_pidx = validate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    gap = train_acc - val_acc
    gap_warning = " ⚠️ OVERFIT" if gap > 10 else ""

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  (gap={gap:.1f}%){gap_warning}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  → Saved best model (val_acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[2].plot(gaps, label='Train-Val Gap', color='red')
axes[2].axhline(y=10, color='orange', linestyle='--', label='Warning threshold (10%)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy Gap (%)')
axes[2].set_title('Overfitting Gap (lower is better)')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v7.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluate on MRI Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))

test_loss, test_acc, test_preds, test_labels, test_pidx = validate(
    model, test_loader, criterion, device
)

print("=" * 60)
print("TEST RESULTS — V7 (MRI Data)")
print("=" * 60)
print(f"Slice-level Accuracy: {test_acc:.2f}%")

label_names = ['Normal', 'Tear']
print("\nSlice-level Classification Report:")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — MRI Test Set, Slice Level (V7)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v7_mri_slice.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Patient-Level Results — MRI Test Set
# ============================================================
print("=" * 60)
print("PATIENT-LEVEL RESULTS — MRI Test Set (Majority Voting)")
print("=" * 60)

mri_p_acc, mri_p_preds, mri_p_labels = patient_level_metrics(
    test_preds, test_labels, test_pidx, test_dataset.df
)

cm_patient = confusion_matrix(mri_p_labels, mri_p_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — MRI Test Set, Patient Level (V7)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v7_mri_patient.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# External Validation — Priyank Saxena Dataset
# ============================================================
print("=" * 60)
print("EXTERNAL VALIDATION — Priyank Saxena Dataset")
print("=" * 60)
print("(Model has NEVER seen any Priyank data during training)")

# Use a simple criterion without class weights for fair eval
eval_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

priyank_loss, priyank_acc, priyank_preds, priyank_labels, priyank_pidx = validate(
    model, priyank_loader, eval_criterion, device
)

print(f"\nSlice-level Accuracy: {priyank_acc:.2f}%")
print("\nSlice-level Classification Report:")
print(classification_report(priyank_labels, priyank_preds, target_names=label_names, digits=3))

cm_priyank = confusion_matrix(priyank_labels, priyank_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_priyank, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Priyank External Validation, Slice Level (V7)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v7_priyank_slice.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Patient-Level Results — Priyank External Validation
# ============================================================
print("=" * 60)
print("PATIENT-LEVEL RESULTS — Priyank External Validation")
print("=" * 60)

priyank_p_acc, priyank_p_preds, priyank_p_labels = patient_level_metrics(
    priyank_preds, priyank_labels, priyank_pidx, priyank_dataset.df
)

cm_priyank_patient = confusion_matrix(priyank_p_labels, priyank_p_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_priyank_patient, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Priyank External Validation, Patient Level (V7)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v7_priyank_patient.png', dpi=150)
plt.show()

# Summary comparison
print("\n" + "=" * 60)
print("V7 SUMMARY — Single Split")
print("=" * 60)
print(f"  MRI Test Set (same source):     {mri_p_acc:.1f}% patient-level accuracy")
print(f"  Priyank (external validation):  {priyank_p_acc:.1f}% patient-level accuracy")
print(f"\n  ℹ️  Priyank results show generalization to a completely unseen MRI source.")
print(f"  ℹ️  Note: Priyank is ~95% Tear, so Normal recall may be unreliable.")

---
## Part B: 5-Fold Stratified Cross-Validation (MRI) + Priyank External Validation

In [ ]:
# ============================================================
# K-Fold CV on MRI Data + Priyank External Validation per Fold
# ============================================================
print(f"\n{'='*60}")
print(f"{N_FOLDS}-FOLD STRATIFIED CROSS-VALIDATION (MRI Data)")
print(f"+ External Validation on Priyank Saxena per fold")
print(f"{'='*60}\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
all_fold_preds = []
all_fold_labels = []
all_fold_priyank_preds = []
all_fold_priyank_labels = []

# Create Priyank dataset once (same for all folds)
priyank_cv_dataset = ACLSliceDataset(priyank_df, DATA_DIR, val_transform)
priyank_cv_loader = DataLoader(priyank_cv_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

for fold, (train_idx, val_idx) in enumerate(skf.split(mri_df, mri_df['label_binary'])):
    print(f"\n{'─'*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'─'*50}")

    fold_train_df = mri_df.iloc[train_idx]
    fold_val_df = mri_df.iloc[val_idx]

    print(f"  Train: {len(fold_train_df)} patients ({fold_train_df['label_name_binary'].value_counts().to_dict()})")
    print(f"  Val:   {len(fold_val_df)} patients ({fold_val_df['label_name_binary'].value_counts().to_dict()})")

    fold_train_dataset = ACLSliceDataset(fold_train_df, DATA_DIR, train_transform, augment_intensity=True)
    fold_val_dataset = ACLSliceDataset(fold_val_df, DATA_DIR, val_transform)

    fold_sampler = make_class_balanced_sampler(fold_train_dataset)

    fold_train_loader = DataLoader(
        fold_train_dataset, batch_size=BATCH_SIZE, sampler=fold_sampler, num_workers=2
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    print("  Creating model...")
    fold_model = create_model()

    fold_train_labels = fold_train_dataset.get_labels()
    fold_class_counts = np.bincount(fold_train_labels)
    fold_class_weights = 1.0 / fold_class_counts
    fold_cw_tensor = torch.tensor(fold_class_weights, dtype=torch.float32).to(device)

    fold_criterion = nn.CrossEntropyLoss(
        weight=fold_cw_tensor, label_smoothing=LABEL_SMOOTHING
    )
    fold_optimizer = optim.AdamW(
        fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    fold_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        fold_optimizer, T_0=10, T_mult=2
    )

    best_fold_acc = 0
    fold_patience = 0
    best_fold_state = None

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_epoch(
            fold_model, fold_train_loader, fold_criterion, fold_optimizer, device
        )
        val_loss, val_acc, _, _, _ = validate(
            fold_model, fold_val_loader, fold_criterion, device
        )
        fold_scheduler.step()

        if val_acc > best_fold_acc:
            best_fold_acc = val_acc
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} (best val_acc: {best_fold_acc:.2f}%)")
                break

    # Load best model for this fold
    fold_model.load_state_dict(best_fold_state)

    # ---- Evaluate on MRI validation fold ----
    _, fold_acc, fold_preds, fold_labels, fold_pidx = validate(
        fold_model, fold_val_loader, fold_criterion, device
    )

    print(f"\n  MRI Val Results:")
    mri_p_acc_fold, fp_preds, fp_labels = patient_level_metrics(
        fold_preds, fold_labels, fold_pidx, fold_val_dataset.df, verbose=True
    )

    # ---- Evaluate on Priyank (external validation) ----
    eval_criterion_fold = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    _, priyank_fold_acc, priyank_fold_preds, priyank_fold_labels, priyank_fold_pidx = validate(
        fold_model, priyank_cv_loader, eval_criterion_fold, device
    )

    print(f"\n  Priyank External Validation:")
    priyank_p_acc_fold, pfp_preds, pfp_labels = patient_level_metrics(
        priyank_fold_preds, priyank_fold_labels, priyank_fold_pidx, priyank_cv_dataset.df, verbose=True
    )

    fold_results.append({
        'fold': fold + 1,
        'mri_slice_acc': fold_acc,
        'mri_patient_acc': mri_p_acc_fold,
        'priyank_slice_acc': priyank_fold_acc,
        'priyank_patient_acc': priyank_p_acc_fold,
        'best_epoch': epoch + 1 - PATIENCE if fold_patience >= PATIENCE else epoch + 1,
    })

    all_fold_preds.extend(fp_preds)
    all_fold_labels.extend(fp_labels)
    all_fold_priyank_preds.extend(pfp_preds)
    all_fold_priyank_labels.extend(pfp_labels)

    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# K-Fold Summary
# ============================================================
print(f"\n{'='*60}")
print(f"K-FOLD CROSS-VALIDATION SUMMARY")
print(f"{'='*60}")

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

print(f"\n{'─'*50}")
print(f"MRI (Same Source):")
print(f"  Mean Slice-Level Accuracy:   {results_df['mri_slice_acc'].mean():.2f}% ± {results_df['mri_slice_acc'].std():.2f}%")
print(f"  Mean Patient-Level Accuracy: {results_df['mri_patient_acc'].mean():.2f}% ± {results_df['mri_patient_acc'].std():.2f}%")

print(f"\nPriyank Saxena (External Validation):")
print(f"  Mean Slice-Level Accuracy:   {results_df['priyank_slice_acc'].mean():.2f}% ± {results_df['priyank_slice_acc'].std():.2f}%")
print(f"  Mean Patient-Level Accuracy: {results_df['priyank_patient_acc'].mean():.2f}% ± {results_df['priyank_patient_acc'].std():.2f}%")

print(f"\n{'─'*50}")
print("Overall MRI Patient-Level Report (all folds combined):")
print(classification_report(
    all_fold_labels, all_fold_preds,
    target_names=label_names, digits=3, zero_division=0
))

print("Overall Priyank Patient-Level Report (all folds combined):")
print("⚠️  Note: Priyank data is evaluated N_FOLDS times (once per fold model)")
print(classification_report(
    all_fold_priyank_labels, all_fold_priyank_preds,
    target_names=label_names, digits=3, zero_division=0
))

In [ ]:
# ============================================================
# K-Fold Visualization
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

x = results_df['fold']
width = 0.2

# Plot 1: All accuracies per fold
axes[0].bar(x - 1.5*width, results_df['mri_slice_acc'], width=width, label='MRI Slice', color='steelblue')
axes[0].bar(x - 0.5*width, results_df['mri_patient_acc'], width=width, label='MRI Patient', color='coral')
axes[0].bar(x + 0.5*width, results_df['priyank_slice_acc'], width=width, label='Priyank Slice', color='mediumpurple')
axes[0].bar(x + 1.5*width, results_df['priyank_patient_acc'], width=width, label='Priyank Patient', color='gold')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy per Fold (V7)')
axes[0].legend(fontsize=8)
axes[0].set_xticks(x)
axes[0].grid(axis='y')

# Plot 2: MRI confusion matrix (all folds)
cm_mri_all = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_mri_all, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'MRI — {N_FOLDS}-Fold CV Patient-Level CM')

# Plot 3: Priyank confusion matrix (all folds combined)
cm_priyank_all = confusion_matrix(all_fold_priyank_labels, all_fold_priyank_preds)
sns.heatmap(cm_priyank_all, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title(f'Priyank External Val — {N_FOLDS}-Fold Patient-Level CM')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/kfold_results_v7.png', dpi=150)
plt.show()